In [1]:
! pip install transformers==4.35.2

In [2]:
# translator_lib.py
# transformers==4.35.2 전제 (DynamicCache 사용 안 함)

from dataclasses import dataclass
from typing import Tuple, List, Literal

import torch
import torch.nn as nn
import torch.nn.functional as F

KVKind = Literal["k", "v"]


# ---------------------------
# KV pack/unpack helpers
# ---------------------------

@dataclass
class ModelKVSpec:
    n_layers: int
    n_heads: int
    head_dim: int
    hidden_size: int  # n_heads * head_dim


def pack_past_key_values(
    past_key_values: Tuple[Tuple[torch.Tensor, torch.Tensor], ...]
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    HF GPT2 past_key_values:
      tuple(n_layers) of (k, v)
      k,v: [B, n_heads, S, head_dim]
    return:
      K,V: [B, S, L, hidden_size]  where hidden_size=n_heads*head_dim
    """
    keys: List[torch.Tensor] = []
    values: List[torch.Tensor] = []

    for (k, v) in past_key_values:
        # [B, nH, S, Hd] -> [B, S, nH, Hd] -> [B, S, nH*Hd]
        k2 = k.permute(0, 2, 1, 3).contiguous().view(k.size(0), k.size(2), -1)
        v2 = v.permute(0, 2, 1, 3).contiguous().view(v.size(0), v.size(2), -1)
        keys.append(k2)
        values.append(v2)

    K = torch.stack(keys, dim=2)   # [B, S, L, D]
    V = torch.stack(values, dim=2) # [B, S, L, D]
    return K, V


def unpack_past_key_values(
    K: torch.Tensor,
    V: torch.Tensor,
    spec: ModelKVSpec
) -> Tuple[Tuple[torch.Tensor, torch.Tensor], ...]:
    """
    K,V: [B, S, L, hidden_size]
    return HF past_key_values tuple:
      k,v: [B, n_heads, S, head_dim]
    """
    B, S, L, D = K.shape
    assert L == spec.n_layers, (L, spec.n_layers)
    assert D == spec.hidden_size, (D, spec.hidden_size)

    out = []
    for layer in range(L):
        k = K[:, :, layer, :].contiguous().view(B, S, spec.n_heads, spec.head_dim).permute(0, 2, 1, 3).contiguous()
        v = V[:, :, layer, :].contiguous().view(B, S, spec.n_heads, spec.head_dim).permute(0, 2, 1, 3).contiguous()
        out.append((k, v))
    return tuple(out)


# ---------------------------
# Cross-attention translator blocks (shared)
# ---------------------------

class CrossAttnBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.ln_q = nn.LayerNorm(d_model)
        self.ln_kv = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, int(d_model * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(d_model * mlp_ratio), d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mem: torch.Tensor) -> torch.Tensor:
        # x:   [B, S, d_model] (query source)
        # mem: [B, S, d_model] (key/value source)
        q = self.ln_q(x)
        kv = self.ln_kv(mem)

        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.mlp(self.ln2(x)))
        return x


# ---------------------------
# Adapters where:
# - Cross-attention blocks are shared between K and V
# - ONLY projections (linear transforms) differ for K vs V
# ---------------------------

class LocalToSharedAdapterKV(nn.Module):
    """
    T[mi -> Σ] for either K or V:
      input:  [B,S,L,Di]
      output: [B,S,Q]

    K/V share cross-attn blocks, but use different input/output projections.
    """
    def __init__(
        self,
        n_layers: int,
        d_in: int,
        q_dim: int,
        d_model: int = 256,
        n_heads: int = 8,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.n_layers = n_layers
        self.d_in = d_in
        self.q_dim = q_dim
        self.d_model = d_model

        # 공유: LN (projection만 분리한다는 가정)
        self.in_ln = nn.LayerNorm(d_in)
        # 분리: input projection (K vs V)
        self.in_proj_k = nn.Linear(d_in, d_model)
        self.in_proj_v = nn.Linear(d_in, d_model)

        # 공유: cross-attn blocks
        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])

        # 공유: output LN
        self.out_ln = nn.LayerNorm(n_layers * d_model)
        # 분리: output projection (K vs V)
        self.out_proj_k = nn.Linear(n_layers * d_model, q_dim)
        self.out_proj_v = nn.Linear(n_layers * d_model, q_dim)

        self.act = nn.GELU()

    def forward(self, x: torch.Tensor, kind: KVKind) -> torch.Tensor:
        # x: [B,S,L,Di]
        B, S, L, Di = x.shape
        assert L == self.n_layers and Di == self.d_in

        x = self.in_ln(x)
        if kind == "k":
            h = self.act(self.in_proj_k(x))
        elif kind == "v":
            h = self.act(self.in_proj_v(x))
        else:
            raise ValueError(f"kind must be 'k' or 'v', got {kind}")

        # h: [B,S,L,d_model]
        cur = h[:, :, 0, :]  # seed: first layer cache
        outs = []

        for layer_idx, blk in enumerate(self.blocks):
            mem = h[:, :, layer_idx, :]
            cur = blk(cur, mem)
            outs.append(cur)

        cat = torch.cat(outs, dim=-1)  # [B,S,L*d_model]
        cat = self.out_ln(cat)

        if kind == "k":
            y = self.act(self.out_proj_k(cat))
        else:
            y = self.act(self.out_proj_v(cat))
        return y  # [B,S,Q]


class SharedToLocalAdapterKV(nn.Module):
    """
    T[Σ -> mi] for either K or V:
      input:  [B,S,Q]
      output: [B,S,L,Di]

    K/V share cross-attn blocks, but use different input/output projections.
    """
    def __init__(
        self,
        n_layers: int,
        q_dim: int,
        d_out: int,
        d_model: int = 256,
        n_heads: int = 8,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        assert q_dim % n_layers == 0, f"q_dim({q_dim}) must be divisible by n_layers({n_layers})"

        self.n_layers = n_layers
        self.q_dim = q_dim
        self.d_out = d_out
        self.d_model = d_model

        self.q_per_layer = q_dim // n_layers

        # 공유 LN
        self.in_ln = nn.LayerNorm(self.q_per_layer)
        # 분리 input projection
        self.in_proj_k = nn.Linear(self.q_per_layer, d_model)
        self.in_proj_v = nn.Linear(self.q_per_layer, d_model)

        # 공유 cross-attn blocks
        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])

        # 공유 LN
        self.out_ln = nn.LayerNorm(n_layers * d_model)
        # 분리 output projection
        self.out_proj_k = nn.Linear(n_layers * d_model, n_layers * d_out)
        self.out_proj_v = nn.Linear(n_layers * d_model, n_layers * d_out)

        self.act = nn.GELU()

    def forward(self, x: torch.Tensor, kind: KVKind) -> torch.Tensor:
        # x: [B,S,Q]
        B, S, Q = x.shape
        assert Q == self.q_dim

        x = x.view(B, S, self.n_layers, self.q_per_layer)  # [B,S,L,Q//L]
        x = self.in_ln(x)

        if kind == "k":
            h = self.act(self.in_proj_k(x))
        elif kind == "v":
            h = self.act(self.in_proj_v(x))
        else:
            raise ValueError(f"kind must be 'k' or 'v', got {kind}")

        # h: [B,S,L,d_model]
        cur = h[:, :, 0, :]
        outs = []
        for layer_idx, blk in enumerate(self.blocks):
            mem = h[:, :, layer_idx, :]
            cur = blk(cur, mem)
            outs.append(cur)

        cat = torch.cat(outs, dim=-1)  # [B,S,L*d_model]
        cat = self.out_ln(cat)

        if kind == "k":
            y = self.act(self.out_proj_k(cat))
        else:
            y = self.act(self.out_proj_v(cat))

        y = y.view(B, S, self.n_layers, self.d_out)
        return y  # [B,S,L,d_out]


# ---------------------------
# 2-model shared-space translator: A <-> Σ <-> B
# (Cross-attn shared between K/V inside each adapter; projections differ)
# ---------------------------

class SharedSpaceKVTranslator(nn.Module):
    def __init__(
        self,
        a_layers: int, a_hidden: int,
        b_layers: int, b_hidden: int,
        q_dim: int = 1536,
        d_model: int = 256,
        n_heads: int = 8,
        dropout: float = 0.0,
    ):
        super().__init__()

        # A <-> Σ
        self.A_to_S = LocalToSharedAdapterKV(a_layers, a_hidden, q_dim, d_model, n_heads, dropout=dropout)
        self.S_to_A = SharedToLocalAdapterKV(a_layers, q_dim, a_hidden, d_model, n_heads, dropout=dropout)

        # B <-> Σ
        self.B_to_S = LocalToSharedAdapterKV(b_layers, b_hidden, q_dim, d_model, n_heads, dropout=dropout)
        self.S_to_B = SharedToLocalAdapterKV(b_layers, q_dim, b_hidden, d_model, n_heads, dropout=dropout)

    @torch.no_grad()
    def translate_A_to_B(self, K_A: torch.Tensor, V_A: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        K_star = self.A_to_S(K_A, "k")
        V_star = self.A_to_S(V_A, "v")
        K_B = self.S_to_B(K_star, "k")
        V_B = self.S_to_B(V_star, "v")
        return K_B, V_B

    @torch.no_grad()
    def translate_B_to_A(self, K_B: torch.Tensor, V_B: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        K_star = self.B_to_S(K_B, "k")
        V_star = self.B_to_S(V_B, "v")
        K_A = self.S_to_A(K_star, "k")
        V_A = self.S_to_A(V_star, "v")
        return K_A, V_A

    def recon_loss_A_to_B(self, K_A, V_A, K_B_target, V_B_target) -> torch.Tensor:
        K_star = self.A_to_S(K_A, "k")
        V_star = self.A_to_S(V_A, "v")
        K_B = self.S_to_B(K_star, "k")
        V_B = self.S_to_B(V_star, "v")
        return F.mse_loss(K_B, K_B_target) + F.mse_loss(V_B, V_B_target)

    def recon_loss_B_to_A(self, K_B, V_B, K_A_target, V_A_target) -> torch.Tensor:
        K_star = self.B_to_S(K_B, "k")
        V_star = self.B_to_S(V_B, "v")
        K_A = self.S_to_A(K_star, "k")
        V_A = self.S_to_A(V_star, "v")
        return F.mse_loss(K_A, K_A_target) + F.mse_loss(V_A, V_A_target)

    def forward(self, K_A, V_A, K_B, V_B) -> torch.Tensor:
        return self.recon_loss_A_to_B(K_A, V_A, K_B, V_B) + self.recon_loss_B_to_A(K_B, V_B, K_A, V_A)


def cosine_sim_flat(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8) -> float:
    a1 = a.detach().float().reshape(-1)
    b1 = b.detach().float().reshape(-1)
    denom = (a1.norm() * b1.norm()).clamp_min(eps)
    return float(torch.dot(a1, b1) / denom)

In [3]:
# code1_train_translator.py
# 목표: Cross Attention Translator 학습
# - A=gpt2, B=gpt2-medium
# - WikiText 데이터로 2000 step 학습
# - 매 100 step 마다 validation reconstruction loss 표시
#
# transformers==4.35.2 전제

import math
import random
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast


# -------------------
# 설정 (argparse 금지)
# -------------------
MODEL_A_NAME = "gpt2"
MODEL_B_NAME = "gpt2-medium"

SEED = 42
CONTEXT_LEN = 64          # toy: 짧게
BATCH_SIZE = 2
TRAIN_STEPS = 1000
VAL_EVERY = 100
VAL_BATCHES = 10          # validation은 일부 배치만

LR = 1e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0

# shared space / translator size (toy)
Q_DIM = 1536              # 24로 나누어 떨어짐(12,24 둘 다 OK)
D_MODEL = 256
N_HEADS = 8
DROPOUT = 0.0

SAVE_PATH = "translator_ckpt.pt"


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_lm_blocks(tokenizer, texts, block_size: int):
    # 텍스트 전체를 이어붙여 block으로 자르는 전형적인 방식
    joined = "\n\n".join([t for t in texts if t is not None])
    ids = tokenizer(joined, return_tensors=None)["input_ids"]
    # ids: List[int]
    n_blocks = len(ids) // block_size
    ids = ids[: n_blocks * block_size]
    blocks = [ids[i * block_size : (i + 1) * block_size] for i in range(n_blocks)]
    return blocks


def collate_fn(batch):
    # batch: List[List[int]] length=CONTEXT_LEN
    input_ids = torch.tensor(batch, dtype=torch.long)
    attention_mask = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask}


@torch.no_grad()
def get_kv_from_model(model, input_ids, attention_mask):
    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True, return_dict=True)
    past = out.past_key_values
    K, V = pack_past_key_values(past)
    return K, V


@torch.no_grad()
def eval_recon_loss(translator, modelA, modelB, val_loader, device):
    translator.eval()
    total = 0.0
    n = 0

    for i, batch in enumerate(val_loader):
        if i >= VAL_BATCHES:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        K_A, V_A = get_kv_from_model(modelA, input_ids, attention_mask)
        K_B, V_B = get_kv_from_model(modelB, input_ids, attention_mask)

        loss = translator(K_A, V_A, K_B, V_B)
        total += float(loss.item())
        n += 1

    translator.train()
    return total / max(1, n)


def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_A_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    # datasets
    ds = load_dataset("wikitext", "wikitext-2-raw-v1")
    train_texts = ds["train"]["text"]
    val_texts = ds["validation"]["text"]

    train_blocks = make_lm_blocks(tokenizer, train_texts, CONTEXT_LEN)
    val_blocks = make_lm_blocks(tokenizer, val_texts, CONTEXT_LEN)

    train_loader = DataLoader(train_blocks, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
    val_loader = DataLoader(val_blocks, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, drop_last=False)

    # models (frozen)
    modelA = GPT2LMHeadModel.from_pretrained(MODEL_A_NAME).to(device)
    modelB = GPT2LMHeadModel.from_pretrained(MODEL_B_NAME).to(device)
    modelA.eval()
    modelB.eval()
    for p in modelA.parameters():
        p.requires_grad_(False)
    for p in modelB.parameters():
        p.requires_grad_(False)

    modelA.config.pad_token_id = tokenizer.eos_token_id
    modelB.config.pad_token_id = tokenizer.eos_token_id

    # specs
    a_layers = modelA.config.n_layer
    a_hidden = modelA.config.n_embd
    b_layers = modelB.config.n_layer
    b_hidden = modelB.config.n_embd

    translator = SharedSpaceKVTranslator(
        a_layers=a_layers, a_hidden=a_hidden,
        b_layers=b_layers, b_hidden=b_hidden,
        q_dim=Q_DIM, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT
    ).to(device)
    translator.train()

    opt = torch.optim.AdamW(translator.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # infinite iterator
    train_iter = iter(train_loader)

    for step in range(1, TRAIN_STEPS + 1):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # target KV from frozen LMs
        with torch.no_grad():
            K_A, V_A = get_kv_from_model(modelA, input_ids, attention_mask)
            K_B, V_B = get_kv_from_model(modelB, input_ids, attention_mask)

        opt.zero_grad(set_to_none=True)
        loss = translator(K_A, V_A, K_B, V_B)
        loss.backward()

        nn.utils.clip_grad_norm_(translator.parameters(), GRAD_CLIP_NORM)
        opt.step()

        if step % 10 == 0:
            print(f"[train] step={step:4d} loss={loss.item():.6f}")

        if step % VAL_EVERY == 0:
            val_loss = eval_recon_loss(translator, modelA, modelB, val_loader, device)
            print(f"[valid] step={step:4d} recon_loss={val_loss:.6f}")

    # save
    ckpt = {
        "translator_state": translator.state_dict(),
        "config": {
            "MODEL_A_NAME": MODEL_A_NAME,
            "MODEL_B_NAME": MODEL_B_NAME,
            "Q_DIM": Q_DIM,
            "D_MODEL": D_MODEL,
            "N_HEADS": N_HEADS,
            "DROPOUT": DROPOUT,
            "a_layers": a_layers,
            "a_hidden": a_hidden,
            "b_layers": b_layers,
            "b_hidden": b_hidden,
        },
    }
    torch.save(ckpt, SAVE_PATH)
    print("saved:", SAVE_PATH)


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (2428601 > 1024). Running this sequence through the model will result in indexing errors
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[train] step=  10 loss=6.942757
[train] step=  20 loss=6.684670
[train] step=  30 loss=6.230149
[train] step=  40 loss=6.419217
[train] step=  50 loss=6.265601
[train] step=  60 loss=6.010645
[train] step=  70 loss=6.327296
[train] step=  80 loss=6.102180
[train] step=  90 loss=6.113821
[train] step= 100 loss=5.997253
[valid] step= 100 recon_loss=6.067449
[train] step= 110 loss=6.073685
[train] step= 120 loss=6.085526
[train] step= 130 loss=6.187150
[train] step= 140 loss=5.765280
[train] step= 150 loss=5.999293
[train] step= 160 loss=5.830255
[train] step= 170 loss=5.951156
[train] step= 180 loss=6.191126
[train] step= 190 loss=6.190643
[train] step= 200 loss=5.838481
[valid] step= 200 recon_loss=5.903767
[train] step= 210 loss=5.934766
[train] step= 220 loss=5.889064
[train] step= 230 loss=5.914603
[train] step= 240 loss=5.857466
[train] step= 250 loss=5.784194
[train] step= 260 loss=5.739898
[train] step= 270 loss=5.793192
[train] step= 280 loss=5.754250
[train] step= 290 loss=5.757

In [4]:
# code2_kv_translate_infer.py
# transformers==4.35.2 전제

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast


# -------------------
# 설정 (argparse 금지)
# -------------------
CKPT_PATH = "translator_ckpt.pt"
MAX_NEW_TOKENS = 30

TARGET_PREFIX_TOKENS = 64  # <-- 요청사항: prefix 길이를 64 토큰으로 맞춤

PREFIXES = [
    "Seoul is the capital of",
    "Paris is the capital of",
]


@torch.no_grad()
def extend_prefix_to_len_with_model(model, prefix_ids: torch.Tensor, target_len: int) -> torch.Tensor:
    """
    prefix_ids를 model로 greedy-extend 해서 토큰 길이를 target_len으로 맞춘다.
    - prefix가 이미 target_len 이상이면 truncate
    - 동일 토큰 시퀀스를 A/B 모두에 입력하기 위해, 여기서는 modelA로만 확장
    """
    model.eval()
    if prefix_ids.size(1) >= target_len:
        return prefix_ids[:, :target_len]

    # 한 번에 prefix 전체를 넣어 cache와 마지막 logits를 얻음
    out = model(input_ids=prefix_ids, use_cache=True, return_dict=True)
    past = out.past_key_values
    generated = prefix_ids

    # out.logits는 prefix의 마지막 토큰 위치까지 포함
    logits = out.logits

    while generated.size(1) < target_len:
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)  # greedy
        generated = torch.cat([generated, next_id], dim=1)

        out = model(input_ids=next_id, past_key_values=past, use_cache=True, return_dict=True)
        past = out.past_key_values
        logits = out.logits

    return generated


@torch.no_grad()
def get_past_excluding_last_token(model, input_ids: torch.Tensor):
    """
    past만으로 generate를 시작하려면,
    past가 prefix[:-1]에 대한 캐시여야 하고,
    첫 step 입력으로 prefix[-1]을 넣어서 next token을 예측하게 만드는 게 깔끔합니다.
    """
    assert input_ids.size(1) >= 2, "prefix는 최소 2토큰 이상이어야 합니다."
    out = model(input_ids=input_ids[:, :-1], use_cache=True, return_dict=True)
    return out.past_key_values


@torch.no_grad()
def greedy_generate_from_past(model, tokenizer, prefix_ids, past_excl_last, max_new_tokens: int):
    model.eval()

    input_ids = prefix_ids[:, -1:]     # 첫 입력: prefix 마지막 토큰
    generated = prefix_ids.clone()
    past = past_excl_last

    for _ in range(max_new_tokens):
        out = model(input_ids=input_ids, past_key_values=past, use_cache=True, return_dict=True)
        logits = out.logits[:, -1, :]
        next_id = torch.argmax(logits, dim=-1, keepdim=True)

        generated = torch.cat([generated, next_id], dim=1)
        past = out.past_key_values
        input_ids = next_id

    return tokenizer.decode(generated[0], skip_special_tokens=True)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    cfg = ckpt["config"]

    tokenizer = GPT2TokenizerFast.from_pretrained(cfg["MODEL_A_NAME"])
    tokenizer.pad_token = tokenizer.eos_token

    modelA = GPT2LMHeadModel.from_pretrained(cfg["MODEL_A_NAME"]).to(device).eval()
    modelB = GPT2LMHeadModel.from_pretrained(cfg["MODEL_B_NAME"]).to(device).eval()
    modelA.config.pad_token_id = tokenizer.eos_token_id
    modelB.config.pad_token_id = tokenizer.eos_token_id

    translator = SharedSpaceKVTranslator(
        a_layers=cfg["a_layers"], a_hidden=cfg["a_hidden"],
        b_layers=cfg["b_layers"], b_hidden=cfg["b_hidden"],
        q_dim=cfg["Q_DIM"], d_model=cfg["D_MODEL"], n_heads=cfg["N_HEADS"], dropout=cfg["DROPOUT"]
    ).to(device).eval()
    translator.load_state_dict(ckpt["translator_state"])

    specA = ModelKVSpec(
        n_layers=modelA.config.n_layer,
        n_heads=modelA.config.n_head,
        head_dim=modelA.config.n_embd // modelA.config.n_head,
        hidden_size=modelA.config.n_embd,
    )
    specB = ModelKVSpec(
        n_layers=modelB.config.n_layer,
        n_heads=modelB.config.n_head,
        head_dim=modelB.config.n_embd // modelB.config.n_head,
        hidden_size=modelB.config.n_embd,
    )

    for raw_prefix in PREFIXES:
        print("\n" + "=" * 80)
        print("RAW PREFIX:", raw_prefix)

        # 1) raw prefix -> ids
        prefix_ids = tokenizer(raw_prefix, return_tensors="pt").input_ids.to(device)

        # 2) prefix를 64 토큰으로 확장 (modelB로만 확장해서 동일 토큰 시퀀스를 만듦)
        # prefix_ids = extend_prefix_to_len_with_model(modelB, prefix_ids, TARGET_PREFIX_TOKENS)

        print(f"PREFIX TOKENS: {prefix_ids.size(1)} (target={TARGET_PREFIX_TOKENS})")
        # 너무 길면 출력이 지저분하니 앞부분만 확인용
        print("PREFIX (decoded):", tokenizer.decode(prefix_ids[0][:64], skip_special_tokens=True))

        # 3) original past (excluding last token)
        pastA = get_past_excluding_last_token(modelA, prefix_ids)
        pastB = get_past_excluding_last_token(modelB, prefix_ids)

        K_A, V_A = pack_past_key_values(pastA)
        K_B, V_B = pack_past_key_values(pastB)

        # 4) translate + round-trip cosine
        with torch.no_grad():
            K_A_to_B, V_A_to_B = translator.translate_A_to_B(K_A, V_A)
            K_B_to_A, V_B_to_A = translator.translate_B_to_A(K_B, V_B)

            K_A_round, V_A_round = translator.translate_B_to_A(K_A_to_B, V_A_to_B)  # A -> B -> A
            K_B_round, V_B_round = translator.translate_A_to_B(K_B_to_A, V_B_to_A)  # B -> A -> B

        cos_A_k = cosine_sim_flat(K_A, K_A_round)
        cos_A_v = cosine_sim_flat(V_A, V_A_round)
        cos_B_k = cosine_sim_flat(K_B, K_B_round)
        cos_B_v = cosine_sim_flat(V_B, V_B_round)
        print(f"[cosine] A round-trip: key={cos_A_k:.6f}, val={cos_A_v:.6f}")
        print(f"[cosine] B round-trip: key={cos_B_k:.6f}, val={cos_B_v:.6f}")

        # 5) unpack translated past
        pastA_original = pastA
        pastB_original = pastB
        pastA_from_B = unpack_past_key_values(K_B_to_A, V_B_to_A, specA)  # B_original -> A
        pastB_from_A = unpack_past_key_values(K_A_to_B, V_A_to_B, specB)  # A_original -> B

        # 6) requested generations (4 cases)
        out1 = greedy_generate_from_past(modelA, tokenizer, prefix_ids, pastA_original,  MAX_NEW_TOKENS)
        out2 = greedy_generate_from_past(modelB, tokenizer, prefix_ids, pastB_from_A,   MAX_NEW_TOKENS)
        out3 = greedy_generate_from_past(modelA, tokenizer, prefix_ids, pastA_from_B,   MAX_NEW_TOKENS)
        out4 = greedy_generate_from_past(modelB, tokenizer, prefix_ids, pastB_original, MAX_NEW_TOKENS)

        print("\n[generate] (1) A_original  -> model A")
        print(out1)
        print("\n[generate] (2) A_to_B      -> model B")
        print(out2)
        print("\n[generate] (3) B_to_A      -> model A")
        print(out3)
        print("\n[generate] (4) B_original  -> model B")
        print(out4)


if __name__ == "__main__":
    main()

device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



RAW PREFIX: Seoul is the capital of
PREFIX TOKENS: 6 (target=64)
PREFIX (decoded): Seoul is the capital of
[cosine] A round-trip: key=0.612664, val=0.460024
[cosine] B round-trip: key=0.598024, val=0.387842

[generate] (1) A_original  -> model A
Seoul is the capital of South Korea, and is home to the country's largest and most famous tourist attraction.

The city is also home to the country's largest and

[generate] (2) A_to_B      -> model B
Seoul is the capital of the of the of the of the of the of the of the of the of the of the of the of the of the of the of the of

[generate] (3) B_to_A      -> model A
Seoul is the capital of the of the of the of the of the of the of the of the of the of the of the of the of the of the of the of

[generate] (4) B_original  -> model B
Seoul is the capital of South Korea, and is home to the country's largest concentration of foreign-born residents.

The city's population is estimated at around 1.

RAW PREFIX: Paris is the capital of
PREFIX TOKENS: 